<a href="https://colab.research.google.com/github/Arpitha-malipatil/Crop-Disease-Identification/blob/main/Frontend_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import gc
import pickle
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ All libraries imported successfully.")

In [ ]:
# ── Save CNN ──────────────────────────────────────────────────────────────────
model.save('plant_disease_cnn.h5')

# ── Save SVM, RF, PCA, Encoder ────────────────────────────────────────────────
with open('svm_model.pkl',   'wb') as f: pickle.dump(svm_model, f)
with open('rf_model.pkl',    'wb') as f: pickle.dump(rf_model,  f)
with open('pca_encoder.pkl', 'wb') as f:
    pickle.dump({'pca': pca, 'encoder': encoder}, f)

print("✅ Models saved:")
print("   plant_disease_cnn.h5")
print("   svm_model.pkl  |  rf_model.pkl  |  pca_encoder.pkl")

In [ ]:
def predict_plant_disease(image_path: str, model_type: str = 'cnn') -> dict:
    """
    Predict plant disease from a leaf image.

    Parameters
    ----------
    image_path : str        path to a .jpg / .png leaf image
    model_type : str        'cnn' | 'svm' | 'rf'

    Returns
    -------
    dict  {'predicted_class': str, 'confidence': str}  (confidence only for CNN)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32) / 255.0

    if model_type == 'cnn':
        probs      = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_idx   = int(np.argmax(probs))
        confidence = float(probs[pred_idx])
        label      = encoder.classes_[pred_idx]
        return {"predicted_class": label, "confidence": f"{confidence*100:.2f}%"}

    elif model_type in ('svm', 'rf'):
        flat     = img.flatten().reshape(1, -1)
        flat_pca = pca.transform(flat)
        clf      = svm_model if model_type == 'svm' else rf_model
        pred     = clf.predict(flat_pca)[0]
        return {"predicted_class": encoder.classes_[pred]}

    raise ValueError("model_type must be 'cnn', 'svm', or 'rf'")

print("✅ predict_plant_disease() ready.")
print("   Example: predict_plant_disease('leaf.jpg', model_type='cnn')")